#Mono Lingual

1. Load the train/test split CSV we created
2. Load x-vector, TRILLsson, HuBERT, and Wav2Vec2 features
3. Reconstruct safe IDs so features match the split file
4. Run monolingual experiments:
   - English train/test on English
   - Mandarin train/test on Mandarin
   - Greek train/test on Greek
5. Run each feature in isolation.
6. Run all features fused together.
7. Evaluate all models:
   - SVM RBF
   - SVM Linear
   - Random Forest
   - KNN
   - XGBoost
   - MLP
8. Metrics:
   - Accuracy
   - Balanced accuracy
   - Precision
   - Recall
   - F1
   - ROC-AUC

In [ ]:
# package installation
!pip install -U scikit-learn xgboost pandas numpy tqdm

In [ ]:
# mount google drive

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)
PROJECT = Path("/content/drive/MyDrive/asr project")
FEATURES_ROOT = PROJECT / "features"
SPLIT_ROOT = PROJECT / "splits"

assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert FEATURES_ROOT.exists(), f"Features folder not found: {FEATURES_ROOT}"
assert SPLIT_ROOT.exists(), f"Split folder not found: {SPLIT_ROOT}"

print("Project:", PROJECT)
print("Features root:", FEATURES_ROOT)
print("Split root:", SPLIT_ROOT)

In [ ]:
# Imports

import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

In [ ]:
# load train/test split file
# load file in advance
SPLIT_CSV = SPLIT_ROOT / "FIN_split_70_30_by_language_grouped_by_safe_speaker.csv"
assert SPLIT_CSV.exists(), f"Split CSV not found: {SPLIT_CSV}"
split_df = pd.read_csv(SPLIT_CSV)


required_split_cols = [
    "split",
    "language",
    "file_id",
    "label_name",
    "label",
    "speaker_id",
    "unique_audio_id",
]
# report if any of the columns are missing
missing_cols = [c for c in required_split_cols if c not in split_df.columns]
assert len(missing_cols) == 0, f"Missing split columns: {missing_cols}"
print("Loaded split file:", SPLIT_CSV)
print("Split shape:", split_df.shape)

print("\nSplit counts:")
display(
    split_df
    .groupby(["split", "language", "label_name"])
    .size()
    .reset_index(name="count")
)

print("\nLanguages:")
print(split_df["language"].value_counts())

print("\nLabels:")
print(split_df["label_name"].value_counts())

In [ ]:
# Purpose: Separate the split into training and test rows and check for audio or speaker leakage
# Checking the splots for debugging purposes

train_df = split_df[split_df["split"] == "train"].copy()
test_df = split_df[split_df["split"] == "test"].copy()
print("Train:", train_df.shape)
print("Test:", test_df.shape)

# check if the split files have same unique audio in train and test
unique_overlap = set(train_df["unique_audio_id"]) & set(test_df["unique_audio_id"])
print("Train/test unique_audio_id overlap:", len(unique_overlap))
assert len(unique_overlap) == 0, "Same audio appears in both train and test."

# no same speaker in train and test, within language and label
train_speaker_keys = set(
    train_df["language"].astype(str)
    + "||"
    + train_df["label_name"].astype(str)
    + "||"
    + train_df["speaker_id"].astype(str)
)

test_speaker_keys = set(
    test_df["language"].astype(str)
    + "||"
    + test_df["label_name"].astype(str)
    + "||"
    + test_df["speaker_id"].astype(str)
)

# calculate the speaker overlap between the two based on the keys identified
speaker_overlap = train_speaker_keys & test_speaker_keys
print("Train/test speaker overlap:", len(speaker_overlap))
assert len(speaker_overlap) == 0, "Speaker leakage detected."

In [ ]:

# feature paths for loading
FEATURE_PATHS = {
    # MCI / Patients features
    "xvector_mci": [
        FEATURES_ROOT / "MCI_xvector_patient_only" / "xvector_features_matrix.npz",
        FEATURES_ROOT / "xvector_patient_only" / "xvector_features_matrix.npz",
    ],

    "trillsson_mci": [
        FEATURES_ROOT / "trillsson_patient_only" / "trillsson_features_matrix.npz",
    ],

    "hubert_mci": [
        FEATURES_ROOT / "ssl_patient_only" / "hubert_all_layers_mean_matrix.npz",
    ],

    "wav2vec2_mci": [
        FEATURES_ROOT / "ssl_patient_only" / "wav2vec2_all_layers_mean_matrix.npz",
    ],


    # Control features
    "xvector_control": [
        FEATURES_ROOT / "xvector_patient_only_control" / "xvector_features_matrix.npz",
        FEATURES_ROOT / "control_xvector_patient_only" / "xvector_features_matrix.npz",
    ],

    "trillsson_control": [
        FEATURES_ROOT / "control_trillsson_patient_only" / "trillsson_features_matrix.npz",
    ],

    "hubert_control": [
        FEATURES_ROOT / "control_ssl_patient_only" / "hubert_all_layers_mean_matrix.npz",
    ],

    "wav2vec2_control": [
        FEATURES_ROOT / "control_ssl_patient_only" / "wav2vec2_all_layers_mean_matrix.npz",
    ],
}


def choose_existing_path(candidate_paths, name):
    """
    Get the first existing path from a list of candidates

    Parameters
    candidate_paths: list of paths
    name: string name of the feature file
    Returns
    """
    #iterate over all candidates
    for path in candidate_paths:
        #if exists return
        if Path(path).exists():
            return Path(path)
    # Print missing files
    print(f"\nMissing candidates for {name}:")
    for path in candidate_paths:
        print("  ", path)

    raise FileNotFoundError(f"No valid path found for {name}")


#store them, checked if they exist and then store them
RESOLVED_FEATURE_PATHS = {}

for name, candidates in FEATURE_PATHS.items():
    path = choose_existing_path(candidates, name)
    RESOLVED_FEATURE_PATHS[name] = path
    print(name, "->", path)

In [ ]:
# ============================================================
# 6. Helper functions for language and safe IDs
# ============================================================

def infer_language_from_dataset_or_path(dataset_name, wav_path=""):
    """
    Infer language from dataset or path.

    Parameters:
    dataset_name: str name of dataset
    wav_path: str path to the wav file

    Returns: str English, Mandarin, Greek, Unknown
    """
    #check if any of the key words is there so asign a language
    text = str(dataset_name) + " " + str(wav_path)
    if "Pitt" in text or "Pits" in text or "English" in text:
        return "English"
    if "Greek" in text or "DemCare" in text or "Dem@Care" in text:
        return "Greek"
    if "Mandarin" in text or "Chou" in text or "Chinese" in text:
        return "Mandarin"
    return "Unknown"


def make_safe_speaker_id_from_parts(language, label_name, file_id):
    """
    Create a normalized speaker ID from the language, label,
    and original file ID. There were speakers that are different with the same id on both control and patients.

    English/Pitt:
        006-2, 006-3, 006-4 -> English_MCI_006

    Greek:
        full file_id

    Mandarin:
        006_Daddy, 006_market, 006_park -> Mandarin_MCI_006
        label_name is included because Control 006 and MCI 006 can both exist.
    """
    language = str(language)
    label_name = str(label_name)
    file_id = str(file_id)

    if language == "English":
        # English recording IDs may contain a session or recording suffix,
        # such as "006-2". Remove the suffix to obtain the base speaker ID
        if "-" in file_id:
            base_id = file_id.split("-")[0]
        else:
            base_id = file_id

        # add the language and label to prevent collisions across datasets
        # or diagnostic groups.
        return f"{language}_{label_name}_{base_id}"

    if language == "Greek":
        # Greek file IDs are already speaker-specific, so retain the full ID
        return f"{language}_{label_name}_{file_id}"

    if language == "Mandarin":
        # Mandarin recording IDs may contain a task suffix separated by "_",
        # for example "006_Daddy",
        if "_" in file_id:
            base_id = file_id.split("_")[0]
        elif "-" in file_id:
            base_id = file_id.split("-")[0]
        else:
            base_id = file_id

        # include the label because identical numeric IDs may exist in
        # different diagnostic groups.
        return f"{language}_{label_name}_{base_id}"
    # for unknown or unsupported language
    return f"{language}_{label_name}_{file_id}"


def add_matching_ids(meta):
    """
    Add standardized identifiers used for matching metadata and audio records

     The following columns are added:

    - language:
        Inferred from the dataset name or WAV path when not already present.
    - speaker_id:
        Normalized speaker-level identifier.
    - unique_audio_id:
        Composite identifier for an individual audio record.
    - key:
        Alias of unique_audio_id for matching or joining tables.

    Return: the updated metadata DataFrame.
    """
    meta = meta.copy()

    # infer language only when the input metadata does not already contain
    # a language column
    if "language" not in meta.columns:
        meta["language"] = [
            infer_language_from_dataset_or_path(d, w)
            for d, w in zip(
                meta["dataset"].astype(str),
                meta.get("wav_path", pd.Series([""] * len(meta))).astype(str)
            )
        ]

    # Create a normalized speaker ID for each row.
    meta["speaker_id"] = [
        make_safe_speaker_id_from_parts(language, label_name, file_id)
        for language, label_name, file_id in zip(
            meta["language"],
            meta["label_name"],
            meta["file_id"]
        )
    ]
    # Create an audio-level identifier by combining language, label,
    # normalized speaker ID, and the original file ID
    meta["unique_audio_id"] = (
        meta["language"].astype(str)
        + "||"
        + meta["label_name"].astype(str)
        + "||"
        + meta["speaker_id"].astype(str)
        + "||"
        + meta["file_id"].astype(str)
    )

    # store the unique audio identifier
    meta["key"] = meta["unique_audio_id"].astype(str)

    return meta

In [ ]:
# 7. Load one feature NPZ

def load_feature_npz(npz_path, label, label_name):
    """
    Load feature vectors and metadata from one compressed  .npz file

    Parameters:
        npz_path: Path to the feature NPZ file.
        label: numeric label used when the NPZ file does not contain
        label_name: string label used when the NPZ file does not contain


    Required NPZ arrays
      -------------------
      1) X - Feature matrix. Each row represents one audio recording
      2) dataset - Dataset name for each feature row
      3) file_id Original recording or file identifier for each feature row
    Optional NPZ arrays
     -------------------
      language, label, label_name, speaker_id, unique_audio_id, wav_path,
      feature_path, thrillson_path, xvectors_path, hubert_path wav2vec_path

    Returns:
        meta : pandas.DataFrame Metadata table containing one row per feature vector.
        X : numpy.ndarray Feature matrix converted to ``float32``.
    """

    npz_path = Path(npz_path)

    if not npz_path.exists():
        raise FileNotFoundError(f"Feature file not found: {npz_path}")

    data = np.load(npz_path, allow_pickle=True)

    X = data["X"].astype(np.float32)

    meta = pd.DataFrame({
        "dataset": data["dataset"].astype(str),
        "file_id": data["file_id"].astype(str),
    })

    if "label" in data:
        meta["label"] = data["label"].astype(int)
    else:
        meta["label"] = int(label)

    if "label_name" in data:
        meta["label_name"] = data["label_name"].astype(str)
    else:
        meta["label_name"] = str(label_name)

    if "language" in data:
        meta["language"] = data["language"].astype(str)

    if "speaker_id" in data:
        meta["speaker_id"] = data["speaker_id"].astype(str)

    if "unique_audio_id" in data:
        meta["unique_audio_id"] = data["unique_audio_id"].astype(str)

    if "wav_path" in data:
        meta["wav_path"] = data["wav_path"].astype(str)
    else:
        meta["wav_path"] = ""

    if "feature_path" in data:
        meta["feature_path"] = data["feature_path"].astype(str)
    elif "xvector_path" in data:
        meta["feature_path"] = data["xvector_path"].astype(str)
    elif "trillsson_path" in data:
        meta["feature_path"] = data["trillsson_path"].astype(str)
    elif "hubert_path" in data:
        meta["feature_path"] = data["hubert_path"].astype(str)
    elif "wav2vec2_path" in data:
        meta["feature_path"] = data["wav2vec2_path"].astype(str)
    else:
        meta["feature_path"] = ""

    meta = add_matching_ids(meta)

    assert len(meta) == X.shape[0], f"Metadata rows and X rows mismatch for {npz_path}"

    # Return the aligned metadata table and numerical feature matrix.
    return meta, X

In [ ]:
# Load MCI + Control for each feature type


#group control and mci files per feature type
FEATURE_GROUPS = {
    "xvector": {
        "mci": RESOLVED_FEATURE_PATHS["xvector_mci"],
        "control": RESOLVED_FEATURE_PATHS["xvector_control"],
    },

    "trillsson": {
        "mci": RESOLVED_FEATURE_PATHS["trillsson_mci"],
        "control": RESOLVED_FEATURE_PATHS["trillsson_control"],
    },

    "hubert": {
        "mci": RESOLVED_FEATURE_PATHS["hubert_mci"],
        "control": RESOLVED_FEATURE_PATHS["hubert_control"],
    },

    "wav2vec2": {
        "mci": RESOLVED_FEATURE_PATHS["wav2vec2_mci"],
        "control": RESOLVED_FEATURE_PATHS["wav2vec2_control"],
    },
}


feature_sets = {}

#iterate over each feature group
for feature_type, paths in FEATURE_GROUPS.items():
    print("\n" + "=" * 80)
    print("Loading feature type:", feature_type)
    print("=" * 80)

    #load mci
    mci_meta, X_mci = load_feature_npz(
        paths["mci"],
        label=1,
        label_name="MCI"
    )

    #load control
    control_meta, X_control = load_feature_npz(
        paths["control"],
        label=0,
        label_name="Control"
    )

    assert X_mci.shape[1] == X_control.shape[1], (
        f"Feature dimension mismatch for {feature_type}: "
        f"MCI {X_mci.shape}, Control {X_control.shape}"
    )

    # Combine features and metadata.
    X = np.vstack([X_mci, X_control]).astype(np.float32)
    meta = pd.concat([mci_meta, control_meta], ignore_index=True)

    # Remove duplicate keys if they exist
    if meta["key"].duplicated().any():
        n_dup = meta["key"].duplicated().sum()
        print(f"Warning: {feature_type} has duplicate keys: {n_dup}. Keeping first.")
        keep_idx = ~meta["key"].duplicated(keep="first")
        # Keep feature rows aligned with metadata
        X = X[keep_idx.values]
        meta = meta[keep_idx].reset_index(drop=True)

    # Extract labels
    y = meta["label"].values.astype(int)


    # Complete feature set
    feature_sets[feature_type] = {
        "X": X,
        "y": y,
        "meta": meta,
    }

    print("MCI X:", X_mci.shape)
    print("Control X:", X_control.shape)
    print("Combined X:", X.shape)

    print("\nLabels:")
    print(meta["label_name"].value_counts())

    print("\nLanguages:")
    print(meta["language"].value_counts())

    print("\nUnique keys:", meta["key"].nunique(), "Rows:", len(meta))

In [ ]:
# match one feature set to the fixed split


def match_feature_to_split(feature_obj, split_df, language):
    """
    For a given language and feature type:
        - get train keys from split file
        - get test keys from split file
        - match feature rows by unique_audio_id/key
        - return X_train, y_train, X_test, y_test, metadata

    This is the key part connecting features to your designed split.
    """
    # load the feature matrix, labels, and metadata.
    X = feature_obj["X"]
    y = feature_obj["y"]
    meta = feature_obj["meta"].copy()

    #split based on language selected
    language_split = split_df[
        split_df["language"] == language
    ].copy()

    #split based on training label
    train_keys = set(
        language_split[
            language_split["split"] == "train"
        ]["unique_audio_id"].astype(str)
    )

    #split based on test label
    test_keys = set(
        language_split[
            language_split["split"] == "test"
        ]["unique_audio_id"].astype(str)
    )

    # find feature rows belonging to this split
    train_mask = meta["key"].astype(str).isin(train_keys).values
    test_mask = meta["key"].astype(str).isin(test_keys).values

    # create training subset
    X_train = X[train_mask]
    y_train = y[train_mask]
    meta_train = meta[train_mask].copy()

    # create the test set
    X_test = X[test_mask]
    y_test = y[test_mask]
    meta_test = meta[test_mask].copy()

    # compare expected and matched sample counts.
    expected_train = len(train_keys)
    expected_test = len(test_keys)

    found_train = len(meta_train)
    found_test = len(meta_test)

    print(f"\n{language}")
    print("Expected train from split:", expected_train)
    print("Found train in features:", found_train)
    print("Missing train:", expected_train - found_train)

    print("Expected test from split:", expected_test)
    print("Found test in features:", found_test)
    print("Missing test:", expected_test - found_test)

    # show class counts in each subset
    print("Train labels:", np.bincount(y_train) if len(y_train) else [])
    print("Test labels:", np.bincount(y_test) if len(y_test) else [])


    if found_train == 0 or found_test == 0:
        print("Warning: no matched train or test rows.")

    # return aligned train/test data
    return {
        "X_train": X_train.astype(np.float32),
        "y_train": y_train.astype(int),
        "meta_train": meta_train.reset_index(drop=True),
        "X_test": X_test.astype(np.float32),
        "y_test": y_test.astype(int),
        "meta_test": meta_test.reset_index(drop=True),
    }

In [ ]:

# Initialize models
def make_models(use_pca=True, pca_components=0.95, random_state=42):
    """
    PCA is fitted only on training data inside the pipeline
    """

    #scale features before PCA and scale-sensitive models
    scaled_steps = [
        ("scaler", StandardScaler())
    ]

    #optionally apply pca
    if use_pca:
        scaled_steps.append(
            ("pca", PCA(
                n_components=pca_components,
                random_state=random_state
            ))
        )

    models = {}

    # nonlinear SVM with balanced class weight
    models["SVM_RBF"] = Pipeline(
        scaled_steps + [
            ("clf", SVC(
                kernel="rbf",
                C=1.0,
                gamma="scale",
                class_weight="balanced",
                probability=True,
                random_state=random_state
            ))
        ]
    )

    # Linear SVM, baseline
    models["SVM_Linear"] = Pipeline(
        scaled_steps + [
            ("clf", SVC(
                kernel="linear",
                C=1.0,
                class_weight="balanced",
                probability=True,
                random_state=random_state
            ))
        ]
    )

    # Random Forest initialization, tree-based
    models["RandomForest"] = Pipeline([
        ("clf", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    # Distance-weighted nearest-neighbour model
    models["KNN"] = Pipeline(
        scaled_steps + [
            ("clf", KNeighborsClassifier(
                n_neighbors=5,
                weights="distance"
            ))
        ]
    )

    # Gradient-boosted tree model
    models["XGBoost"] = Pipeline([
        ("clf", XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1
        ))
    ])

    # Neural-network classifier with early stopping
    models["MLP"] = Pipeline(
        scaled_steps + [
            ("clf", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=1e-3,
                max_iter=1000,
                early_stopping=True,
                random_state=random_state
            ))
        ]
    )

    return models

In [ ]:
# Prediction probability helper

def get_positive_scores(model, X_test):
    """
    Return score for class 1 for ROC-AUC.
    """
    if hasattr(model, "predict_proba"):
        # get predicted probabilities
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:
            return proba[:, 1]
    # otherwise if not probabilities get model's decision scores
    if hasattr(model, "decision_function"):
        return model.decision_function(X_test)

    return None

In [ ]:
LANGUAGES = ["English", "Greek", "Mandarin"]
# features
FEATURE_NAMES = ["xvector", "trillsson", "hubert", "wav2vec2"]

In [ ]:
# Build monolingual matched datasets for isolated features

#languages included in experiment
LANGUAGES = ["English", "Greek", "Mandarin"]
# features
FEATURE_NAMES = ["xvector", "trillsson", "hubert", "wav2vec2"]

mono_isolated = {}

# Process each language independently
for language in LANGUAGES:
    mono_isolated[language] = {}
    # Match every feature type to the fixed split
    for feature_name in FEATURE_NAMES:
        print("\n" + "#" * 80)
        print("Matching:", language, feature_name)
        print("#" * 80)
        # Create aligned training and test subsets
        matched = match_feature_to_split(
            feature_obj=feature_sets[feature_name],
            split_df=split_df,
            language=language
        )

        mono_isolated[language][feature_name] = matched

In [ ]:
# Create all-feature fusion per language

def create_fused_for_language(language, mono_isolated, feature_names):
    """
    Fuse features for one language using only samples that exist
    in all feature sets and are in the same train/test split.
    """

    # store fused train and test data
    fused = {}

    # process train and test
    for split_name in ["train", "test"]:
        key_sets = []

        # collect keys from each feature set
        for feature_name in feature_names:
            meta = mono_isolated[language][feature_name][f"meta_{split_name}"]
            key_sets.append(set(meta["key"].astype(str)))

        # keep only recordings available in every feature set
        common_keys = set.intersection(*key_sets)

        print("\n", language, split_name)
        print("Common keys across all features:", len(common_keys))

        X_parts = []
        y_base = None
        meta_base = None
        base_keys = None

        # align each feature set using the common keys
        for feature_name in feature_names:
            obj = mono_isolated[language][feature_name]

            X_split = obj[f"X_{split_name}"]
            y_split = obj[f"y_{split_name}"]
            meta_split = obj[f"meta_{split_name}"].copy()
            meta_split["original_index"] = np.arange(len(meta_split))

            # select rows available in all feature sets
            meta_split = meta_split[
                meta_split["key"].astype(str).isin(common_keys)
            ].copy()

            # remove duplicate recordings
            if meta_split["key"].duplicated().any():
                meta_split = meta_split.drop_duplicates(subset="key", keep="first").copy()

            # sort keys so all feature use the same row order.
            meta_split = meta_split.sort_values("key").reset_index(drop=True)
            indices = meta_split["original_index"].values

            # align features and labels with the sorted metadata
            X_matched = X_split[indices]
            y_matched = y_split[indices]

            meta_split = meta_split.drop(columns=["original_index"])

            # use the first feature set as the alignment reference.
            if y_base is None:
                y_base = y_matched
                meta_base = meta_split.copy()
                base_keys = meta_split["key"].values
            else:
                #  identical key and label order.
                assert np.array_equal(meta_split["key"].values, base_keys), (
                    f"Key order mismatch for {language}, {split_name}, {feature_name}"
                )
                assert np.array_equal(y_matched, y_base), (
                    f"Label mismatch for {language}, {split_name}, {feature_name}"
                )

            print(feature_name, X_matched.shape)

            # add the aligned feature matrix
            X_parts.append(X_matched.astype(np.float32))

        # join feature dimensions
        X_fused = np.concatenate(X_parts, axis=1).astype(np.float32)

        # save the new split
        fused[f"X_{split_name}"] = X_fused
        fused[f"y_{split_name}"] = y_base.astype(int)
        fused[f"meta_{split_name}"] = meta_base.reset_index(drop=True)

    return fused


# store fused datasets by language.
mono_fused = {}

# create fused features for each language
for language in LANGUAGES:
    print("\n" + "=" * 80)
    print("Creating fused features for:", language)
    print("=" * 80)

    mono_fused[language] = create_fused_for_language(
        language=language,
        mono_isolated=mono_isolated,
        feature_names=FEATURE_NAMES
    )

    # show final fused matrix sizes.
    print("Fused train:", mono_fused[language]["X_train"].shape)
    print("Fused test:", mono_fused[language]["X_test"].shape)


In [ ]:
# evaluate multiple models on one fixed train/test split

def evaluate_train_test(
    experiment_name,
    feature_name,
    language,
    X_train,
    y_train,
    X_test,
    y_test,
    use_pca=True,
    pca_components=0.95,
    random_state=42,
):
    """
    Fit on train, evaluate on test.

    Saves:
        accuracy
        balanced_accuracy
        precision
        recall
        f1
        roc_auc

    Also saves class-wise:
        precision_control
        recall_control
        f1_control
        precision_mci
        recall_mci
        f1_mci
    """

    # store one result row per model
    rows = []

    # the experiment settings
    print("\n" + "=" * 80)
    print("Experiment:", experiment_name)
    print("Feature:", feature_name)
    print("Language:", language)
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("Train labels:", np.bincount(y_train) if len(y_train) else [])
    print("Test labels:", np.bincount(y_test) if len(y_test) else [])
    print("PCA:", use_pca, pca_components)
    print("=" * 80)

    # training requires both classes
    if len(np.unique(y_train)) < 2:
        return pd.DataFrame([{
            "experiment": experiment_name,
            "feature_set": feature_name,
            "language": language,
            "model": "",
            "use_pca": use_pca,
            "pca_components": pca_components,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "n_features": X_train.shape[1] if len(X_train) else np.nan,

            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "roc_auc": np.nan,

            "precision_control": np.nan,
            "recall_control": np.nan,
            "f1_control": np.nan,
            "precision_mci": np.nan,
            "recall_mci": np.nan,
            "f1_mci": np.nan,

            "confusion_matrix": "",
            "error": "training set has fewer than two classes",
        }])

    # create fresh model pipelines
    models = make_models(
        use_pca=use_pca,
        pca_components=pca_components,
        random_state=random_state
    )

    # train and evaluate each model
    for model_name, model in models.items():
        try:
            # Fit using training data only
            model.fit(X_train, y_train)

            # predict test labels and class scores
            y_pred = model.predict(X_test)
            y_score = get_positive_scores(model, X_test)

            # calculate ROC-AUC when both test classes exist
            if y_score is not None and len(np.unique(y_test)) == 2:
                roc_auc = roc_auc_score(y_test, y_score)
            else:
                roc_auc = np.nan

            # generate overall and class-specific metrics
            report = classification_report(
                y_test,
                y_pred,
                labels=[0, 1],
                target_names=["Control", "MCI"],
                output_dict=True,
                zero_division=0,
            )

            # store the evaluation results
            row = {
                "experiment": experiment_name,
                "feature_set": feature_name,
                "language": language,
                "model": model_name,
                "use_pca": use_pca,
                "pca_components": pca_components,
                "n_train": len(y_train),
                "n_test": len(y_test),
                "n_features": X_train.shape[1],

                # MCI is the positive class
                "accuracy": accuracy_score(y_test, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
                "recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
                "f1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
                "roc_auc": roc_auc,

                # treat both classes equally
                "precision_macro": report["macro avg"]["precision"],
                "recall_macro": report["macro avg"]["recall"],
                "f1_macro": report["macro avg"]["f1-score"],

                # weight results by class size
                "precision_weighted": report["weighted avg"]["precision"],
                "recall_weighted": report["weighted avg"]["recall"],
                "f1_weighted": report["weighted avg"]["f1-score"],

                # control-class metrics
                "precision_control": report["Control"]["precision"],
                "recall_control": report["Control"]["recall"],
                "f1_control": report["Control"]["f1-score"],

                # MCI-class metrics
                "precision_mci": report["MCI"]["precision"],
                "recall_mci": report["MCI"]["recall"],
                "f1_mci": report["MCI"]["f1-score"],

                # save prediction counts
                "confusion_matrix": str(confusion_matrix(y_test, y_pred, labels=[0, 1]).tolist()),
                "error": "",
            }

            rows.append(row)

        # save the error and continue with other models
        except Exception as e:
            rows.append({
                "experiment": experiment_name,
                "feature_set": feature_name,
                "language": language,
                "model": model_name,
                "use_pca": use_pca,
                "pca_components": pca_components,
                "n_train": len(y_train),
                "n_test": len(y_test),
                "n_features": X_train.shape[1] if len(X_train) else np.nan,

                "accuracy": np.nan,
                "balanced_accuracy": np.nan,
                "precision": np.nan,
                "recall": np.nan,
                "f1": np.nan,
                "roc_auc": np.nan,

                "precision_macro": np.nan,
                "recall_macro": np.nan,
                "f1_macro": np.nan,

                "precision_weighted": np.nan,
                "recall_weighted": np.nan,
                "f1_weighted": np.nan,

                "precision_control": np.nan,
                "recall_control": np.nan,
                "f1_control": np.nan,

                "precision_mci": np.nan,
                "recall_mci": np.nan,
                "f1_mci": np.nan,

                "confusion_matrix": "",
                "error": str(e),
            })

    # return all model results as a DataFrame
    return pd.DataFrame(rows)

In [ ]:
# run monolingual isolated-feature experiments

# store results from every experiment
all_results = []

# evaluate each language separately
for language in LANGUAGES:

    # test each feature representation independently
    for feature_name in FEATURE_NAMES:
        obj = mono_isolated[language][feature_name]

        # load the matched train and test data
        X_train = obj["X_train"]
        y_train = obj["y_train"]
        X_test = obj["X_test"]
        y_test = obj["y_test"]

        # skip datasets with no train or test samples
        if len(y_train) == 0 or len(y_test) == 0:
            print("Skipping empty:", language, feature_name)
            continue

        # evaluate models with PCA
        res_pca = evaluate_train_test(
            experiment_name=f"{language}_{feature_name}_pca",
            feature_name=feature_name,
            language=language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=True,
            pca_components=0.95,
            random_state=42
        )

        # save PCA results
        all_results.append(res_pca)

        # evaluate models without PCA
        res_no_pca = evaluate_train_test(
            experiment_name=f"{language}_{feature_name}_no_pca",
            feature_name=feature_name,
            language=language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=False,
            pca_components=None,
            random_state=42
        )

        # save no-PCA results.
        all_results.append(res_no_pca)


In [ ]:
# ============================================================
# 16. Run monolingual all-feature fusion experiments
# ============================================================

#evaluate every language independently
for language in LANGUAGES:
    # use fused data
    obj = mono_fused[language]

    X_train = obj["X_train"]
    y_train = obj["y_train"]
    X_test = obj["X_test"]
    y_test = obj["y_test"]

    if len(y_train) == 0 or len(y_test) == 0:
        print("Skipping empty fused:", language)
        continue

    # PCA version
    res_pca = evaluate_train_test(
        experiment_name=f"{language}_all_features_fused_pca",
        feature_name="all_features_fused",
        language=language,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        use_pca=True,
        pca_components=0.95,
        random_state=42
    )

    all_results.append(res_pca)

    # No-PCA version
    res_no_pca = evaluate_train_test(
        experiment_name=f"{language}_all_features_fused_no_pca",
        feature_name="all_features_fused",
        language=language,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        use_pca=False,
        pca_components=None,
        random_state=42
    )

    all_results.append(res_no_pca)

In [ ]:
# Combine and save results


# combine all experiment result tables
results_df = pd.concat(all_results, ignore_index=True)

# the output CSV path
RESULTS_PATH = PROJECT / "none_interpretable_model_results_monolingual_train_test_all_features_and_isolated.csv"

# Save all results
results_df.to_csv(RESULTS_PATH, index=False)


print("Saved results to:", RESULTS_PATH)
print("Results shape:", results_df.shape)
print("\nColumns saved:")
print(results_df.columns.tolist())

# show the 50 best results
display(
    results_df.sort_values(
        # rank balanced accuracy first, then use other metrics as tie-breakers
        by=["balanced_accuracy", "f1", "roc_auc", "accuracy"],
        ascending=False
    ).head(50)
)


In [ ]:
# rank results and keep the top 10 for each language
best_per_language = (
    results_df
    .sort_values(
        # use balanced accuracy first, then other metrics as tie-breakers
        by=["balanced_accuracy", "f1", "roc_auc", "accuracy"],
        ascending=False
    )
    .groupby("language")
    .head(10)
    .reset_index(drop=True)
)

display(best_per_language)

# output file path
BEST_PATH = PROJECT / "none_interpretable_model_results_monolingual_best_per_language.csv"

# Save the best results per language
best_per_language.to_csv(BEST_PATH, index=False)

print("Saved best per language:", BEST_PATH)


In [ ]:
# best result per language and feature set


# tank results and keep the top 3 models for each language-feature pair
best_per_language_feature = (
    results_df
    .sort_values(
        # use balanced accuracy first, then other metrics as tie-breakers
        by=["balanced_accuracy", "f1", "roc_auc", "accuracy"],
        ascending=False
    )
    .groupby(["language", "feature_set"])
    .head(3)
    .reset_index(drop=True)
)

display(best_per_language_feature)

# output CSV path
BEST_FEATURE_PATH = PROJECT / "none_interpretable_model_results_monolingual_best_per_language_feature.csv"

# save the top results
best_per_language_feature.to_csv(BEST_FEATURE_PATH, index=False)

print("Saved best per language/feature:", BEST_FEATURE_PATH)



# Multilingual

Train: all languages' train sets together
Test: one language's test set at a time

Experiments:
1. Train English+Greek+Mandarin, test English
2. Train English+Greek+Mandarin, test Greek
3. Train English+Greek+Mandarin, test Mandarin

For:
- xvector
- trillsson
- hubert
- wav2vec2
- all_features_fused

Models:
- SVM RBF
- SVM Linear
- Random Forest
- KNN
- XGBoost
- MLP

Metrics:
- Accuracy
- Balanced accuracy
- Precision
- Recall
- F1
- ROC-AUC

In [ ]:
# Build multilingual train + one-language test datasets
# For isolated feature sets
#
def build_multilingual_train_language_test_isolated(
    mono_isolated,
    feature_name,
    test_language,
    languages,
):
    """
    Build dataset for multilingual scenario using one isolated feature.

    Train:
        all languages' train sets

    Test:
        only test_language test set
    """

    # store training data from all languages
    X_train_parts = []
    y_train_parts = []
    meta_train_parts = []

    # combine each language's training split
    for language in languages:
        obj = mono_isolated[language][feature_name]

        X_train_parts.append(obj["X_train"])
        y_train_parts.append(obj["y_train"])

        # record the original language of each sample
        meta_tmp = obj["meta_train"].copy()
        meta_tmp["source_language"] = language
        meta_train_parts.append(meta_tmp)

    # merge all multilingual training data
    X_train = np.vstack(X_train_parts).astype(np.float32)
    y_train = np.concatenate(y_train_parts).astype(int)
    meta_train = pd.concat(meta_train_parts, ignore_index=True)

    # use one language's fixed test split
    test_obj = mono_isolated[test_language][feature_name]

    X_test = test_obj["X_test"].astype(np.float32)
    y_test = test_obj["y_test"].astype(int)
    meta_test = test_obj["meta_test"].copy()
    meta_test["source_language"] = test_language


    print("\n" + "=" * 80)
    print("Isolated multilingual dataset")
    print("Feature:", feature_name)
    print("Test language:", test_language)
    print("X_train:", X_train.shape)
    print("y_train:", np.bincount(y_train))
    print("X_test:", X_test.shape)
    print("y_test:", np.bincount(y_test))
    print("=" * 80)

    # return the combined training and test data
    return {
        "X_train": X_train,
        "y_train": y_train,
        "meta_train": meta_train,
        "X_test": X_test,
        "y_test": y_test,
        "meta_test": meta_test,
    }


In [ ]:
# build multilingual train + one-language test datasets
# For all-feature fusion

def build_multilingual_train_language_test_fused(
    mono_fused,
    test_language,
    languages,
):
    """
    Build dataset for multilingual scenario using all fused features.

    Train:
        all languages' fused train sets

    Test:
        only test_language fused test set
    """

    # store training data from all languages
    X_train_parts = []
    y_train_parts = []
    meta_train_parts = []

    # combine each language's fused training split
    for language in languages:
        obj = mono_fused[language]

        X_train_parts.append(obj["X_train"])
        y_train_parts.append(obj["y_train"])

        # record each sample's source language
        meta_tmp = obj["meta_train"].copy()
        meta_tmp["source_language"] = language
        meta_train_parts.append(meta_tmp)

    # merge all multilingual training data
    X_train = np.vstack(X_train_parts).astype(np.float32)
    y_train = np.concatenate(y_train_parts).astype(int)
    meta_train = pd.concat(meta_train_parts, ignore_index=True)

    # use one language's fused test split
    test_obj = mono_fused[test_language]

    X_test = test_obj["X_test"].astype(np.float32)
    y_test = test_obj["y_test"].astype(int)
    meta_test = test_obj["meta_test"].copy()
    meta_test["source_language"] = test_language

    print("\n" + "=" * 80)
    print("Fused multilingual dataset")
    print("Test language:", test_language)
    print("X_train:", X_train.shape)
    print("y_train:", np.bincount(y_train))
    print("X_test:", X_test.shape)
    print("y_test:", np.bincount(y_test))
    print("=" * 80)

    # Return the combined training and test data.
    return {
        "X_train": X_train,
        "y_train": y_train,
        "meta_train": meta_train,
        "X_test": X_test,
        "y_test": y_test,
        "meta_test": meta_test,
    }


In [ ]:
# safety check for multilingual train and test leakage

def check_multilingual_no_leakage(dataset_obj, test_language):
    """
    Checks:
        - no same unique_audio_id in train and test
        - no same speaker_id within language/label in train and test
    """

    # copy train and test metadata
    meta_train = dataset_obj["meta_train"].copy()
    meta_test = dataset_obj["meta_test"].copy()

    # collect unique audio keys
    train_audio_keys = set(meta_train["key"].astype(str))
    test_audio_keys = set(meta_test["key"].astype(str))

    # find recordings present in both sets
    audio_overlap = train_audio_keys & test_audio_keys

    # show audio leakage results
    print("\nLeakage check for test language:", test_language)
    print("Audio overlap:", len(audio_overlap))

    # show examples when overlap exists
    if len(audio_overlap) > 0:
        print("Example overlapping audio keys:")
        print(list(audio_overlap)[:10])

    # stop when audio leakage is found
    assert len(audio_overlap) == 0, "Audio leakage detected."

    # create unique train speaker keys
    train_speaker_keys = set(
        meta_train["language"].astype(str)
        + "||"
        + meta_train["label_name"].astype(str)
        + "||"
        + meta_train["speaker_id"].astype(str)
    )

    # create unique test speaker keys
    test_speaker_keys = set(
        meta_test["language"].astype(str)
        + "||"
        + meta_test["label_name"].astype(str)
        + "||"
        + meta_test["speaker_id"].astype(str)
    )

    # find speakers present in both sets
    speaker_overlap = train_speaker_keys & test_speaker_keys

    # show speaker leakage results
    print("Speaker overlap:", len(speaker_overlap))

    # show examples when overlap exists
    if len(speaker_overlap) > 0:
        print("Example overlapping speaker keys:")
        print(list(speaker_overlap)[:10])

    # stop when speaker leakage is found
    assert len(speaker_overlap) == 0, "Speaker leakage detected."


In [ ]:
# ============================================================
# 25. Run multilingual experiments for isolated features
# ============================================================

multilingual_results = []

for test_language in LANGUAGES:
    for feature_name in FEATURE_NAMES:

        dataset_obj = build_multilingual_train_language_test_isolated(
            mono_isolated=mono_isolated,
            feature_name=feature_name,
            test_language=test_language,
            languages=LANGUAGES,
        )

        check_multilingual_no_leakage(
            dataset_obj=dataset_obj,
            test_language=test_language,
        )

        X_train = dataset_obj["X_train"]
        y_train = dataset_obj["y_train"]
        X_test = dataset_obj["X_test"]
        y_test = dataset_obj["y_test"]

        # PCA version
        res_pca = evaluate_train_test(
            experiment_name=f"multilingual_train_all_test_{test_language}_{feature_name}_pca",
            feature_name=feature_name,
            language=test_language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=True,
            pca_components=0.95,
            random_state=42,
        )

        res_pca["training_setup"] = "multilingual_train_all_languages"
        res_pca["test_language"] = test_language

        multilingual_results.append(res_pca)

        # No-PCA version
        res_no_pca = evaluate_train_test(
            experiment_name=f"multilingual_train_all_test_{test_language}_{feature_name}_no_pca",
            feature_name=feature_name,
            language=test_language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=False,
            pca_components=None,
            random_state=42,
        )

        res_no_pca["training_setup"] = "multilingual_train_all_languages"
        res_no_pca["test_language"] = test_language

        multilingual_results.append(res_no_pca)

In [ ]:

#  multilingual experiments for each feature

# store all multilingual results
multilingual_results = []

# test each language separately
for test_language in LANGUAGES:

    # evaluate each feature type
    for feature_name in FEATURE_NAMES:

        # build multilingual train and language-specific test data
        dataset_obj = build_multilingual_train_language_test_isolated(
            mono_isolated=mono_isolated,
            feature_name=feature_name,
            test_language=test_language,
            languages=LANGUAGES,
        )

        # check for train and test leakage
        check_multilingual_no_leakage(
            dataset_obj=dataset_obj,
            test_language=test_language,
        )

        X_train = dataset_obj["X_train"]
        y_train = dataset_obj["y_train"]
        X_test = dataset_obj["X_test"]
        y_test = dataset_obj["y_test"]

        # evaluate with pca
        res_pca = evaluate_train_test(
            experiment_name=f"multilingual_train_all_test_{test_language}_{feature_name}_pca",
            feature_name=feature_name,
            language=test_language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=True,
            pca_components=0.95,
            random_state=42,
        )

        res_pca["training_setup"] = "multilingual_train_all_languages"
        res_pca["test_language"] = test_language

        # save pca results
        multilingual_results.append(res_pca)

        # evaluate without pca
        res_no_pca = evaluate_train_test(
            experiment_name=f"multilingual_train_all_test_{test_language}_{feature_name}_no_pca",
            feature_name=feature_name,
            language=test_language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=False,
            pca_components=None,
            random_state=42,
        )

        res_no_pca["training_setup"] = "multilingual_train_all_languages"
        res_no_pca["test_language"] = test_language

        # save no pca results
        multilingual_results.append(res_no_pca)

In [ ]:
# combine and save multilingual results

# combine all multilingual result tables
multilingual_results_df = pd.concat(
    multilingual_results,
    ignore_index=True
)

# define the output csv path
MULTILINGUAL_RESULTS_PATH = PROJECT / "none_interpretable_model_results_multilingual_train_all_test_each_language.csv"

# save the combined results
multilingual_results_df.to_csv(MULTILINGUAL_RESULTS_PATH, index=False)

print("Saved multilingual results to:", MULTILINGUAL_RESULTS_PATH)
print("Results shape:", multilingual_results_df.shape)
print("\nColumns:")
print(multilingual_results_df.columns.tolist())

display(
    multilingual_results_df.sort_values(
        # rank by balanced accuracy first
        by=["balanced_accuracy", "f1", "roc_auc", "accuracy"],
        ascending=False
    ).head(50)
)


# Cross


## CROSS-LINGUAL MODELING

* Train on 2 languages and test on the held-out language.

Experiments:
  * Train English + Greek      -> Test Mandarin
  * Train English + Mandarin   -> Test Greek
  * Train Greek + Mandarin     -> Test English

Feature sets:
  * xvector
  * trillsson
  * hubert
  * wav2vec2
  * all_features_fused

Models:
  * SVM RBF
  * SVM Linear
  * Random Forest
  * KNN
  * XGBoost
  * MLP

Metrics come from evaluate_train_test():
  * accuracy
  * balanced_accuracy
  * precision
  * recall
  * f1
  * roc_auc
  * plus class-wise metrics if you replaced the function earlier

In [ ]:
# define each cross-lingual experiment
CROSS_LINGUAL_SETUPS = [
    {
        # train on english and greek
        "train_languages": ["English", "Greek"],

        # test on unseen mandarin data
        "test_language": "Mandarin",

        # name used to identify the setup
        "setup_name": "train_English_Greek_test_Mandarin",
    },
    {
        # train on english and mandarin
        "train_languages": ["English", "Mandarin"],

        # test on unseen greek data
        "test_language": "Greek",

        # name used to identify the setup
        "setup_name": "train_English_Mandarin_test_Greek",
    },
    {
        # train on greek and mandarin
        "train_languages": ["Greek", "Mandarin"],

        # test on unseen english data
        "test_language": "English",

        # name used to identify the setup
        "setup_name": "train_Greek_Mandarin_test_English",
    },
]

for setup in CROSS_LINGUAL_SETUPS:
    print(setup)



In [ ]:
#build cross-lingual dataset for isolated features

def build_crosslingual_train_test_isolated(
    mono_isolated,
    feature_name,
    train_languages,
    test_language,
):
    """
    Cross-lingual setup for one isolated feature.

    Train:
        train split from two languages

    Test:
        test split from held-out language
    """

    X_train_parts = []
    y_train_parts = []
    meta_train_parts = []

    # combine training splits from selected languages
    for language in train_languages:
        obj = mono_isolated[language][feature_name]

        X_train_parts.append(obj["X_train"])
        y_train_parts.append(obj["y_train"])

        # record the source language
        meta_tmp = obj["meta_train"].copy()
        meta_tmp["source_language"] = language
        meta_train_parts.append(meta_tmp)

    # merge all training data
    X_train = np.vstack(X_train_parts).astype(np.float32)
    y_train = np.concatenate(y_train_parts).astype(int)
    meta_train = pd.concat(meta_train_parts, ignore_index=True)

    # use the held-out language test split
    test_obj = mono_isolated[test_language][feature_name]

    X_test = test_obj["X_test"].astype(np.float32)
    y_test = test_obj["y_test"].astype(int)
    meta_test = test_obj["meta_test"].copy()
    meta_test["source_language"] = test_language


    print("\n" + "=" * 80)
    print("Cross-lingual isolated dataset")
    print("Feature:", feature_name)
    print("Train languages:", train_languages)
    print("Test language:", test_language)
    print("X_train:", X_train.shape)
    print("y_train:", np.bincount(y_train))
    print("X_test:", X_test.shape)
    print("y_test:", np.bincount(y_test))
    print("=" * 80)

    # return train and test data
    return {
        "X_train": X_train,
        "y_train": y_train,
        "meta_train": meta_train,
        "X_test": X_test,
        "y_test": y_test,
        "meta_test": meta_test,
    }


In [ ]:
# build cross-lingual dataset for all-feature fusion

def build_crosslingual_train_test_fused(
    mono_fused,
    train_languages,
    test_language,
):
    """
    Cross-lingual setup for all fused features.

    Train:
        train split from two languages

    Test:
        test split from held-out language
    """
    X_train_parts = []
    y_train_parts = []
    meta_train_parts = []

    # combine fused training splits
    for language in train_languages:
        obj = mono_fused[language]

        X_train_parts.append(obj["X_train"])
        y_train_parts.append(obj["y_train"])

        # record the source language
        meta_tmp = obj["meta_train"].copy()
        meta_tmp["source_language"] = language
        meta_train_parts.append(meta_tmp)

    # merge all training data
    X_train = np.vstack(X_train_parts).astype(np.float32)
    y_train = np.concatenate(y_train_parts).astype(int)
    meta_train = pd.concat(meta_train_parts, ignore_index=True)

    # use the held-out language test split
    test_obj = mono_fused[test_language]

    X_test = test_obj["X_test"].astype(np.float32)
    y_test = test_obj["y_test"].astype(int)
    meta_test = test_obj["meta_test"].copy()
    meta_test["source_language"] = test_language


    print("\n" + "=" * 80)
    print("Cross-lingual fused dataset")
    print("Train languages:", train_languages)
    print("Test language:", test_language)
    print("X_train:", X_train.shape)
    print("y_train:", np.bincount(y_train))
    print("X_test:", X_test.shape)
    print("y_test:", np.bincount(y_test))
    print("=" * 80)

    # return train and test data
    return {
        "X_train": X_train,
        "y_train": y_train,
        "meta_train": meta_train,
        "X_test": X_test,
        "y_test": y_test,
        "meta_test": meta_test,
    }


In [ ]:
# Cross-lingual leakage checks

def check_crosslingual_no_leakage(dataset_obj, train_languages, test_language):
    """
    Checks:
        - no same audio in train and test
        - no same speaker in train and test

    In true cross-lingual setup, this should be zero because
    test language is not part of train languages.
    """

    # copy train and test metadata
    meta_train = dataset_obj["meta_train"].copy()
    meta_test = dataset_obj["meta_test"].copy()

    print("\nLeakage check")
    print("Train languages:", train_languages)
    print("Test language:", test_language)
    print("Train language counts:")
    print(meta_train["language"].value_counts())
    print("Test language counts:")
    print(meta_test["language"].value_counts())

    # confirm the held-out language is not in training
    train_langs_found = set(meta_train["language"].astype(str))
    assert test_language not in train_langs_found, (
        f"Test language {test_language} appears in training data."
    )

    # collect unique audio keys
    train_audio_keys = set(meta_train["key"].astype(str))
    test_audio_keys = set(meta_test["key"].astype(str))

    # find audio present in both sets
    audio_overlap = train_audio_keys & test_audio_keys

    # show audio overlap count
    print("Audio overlap:", len(audio_overlap))

    # show examples when overlap exists
    if len(audio_overlap) > 0:
        print("Example overlapping audio keys:")
        print(list(audio_overlap)[:10])

    # stop when audio leakage is found
    assert len(audio_overlap) == 0, "Audio leakage detected."

    # create unique training speaker keys
    train_speaker_keys = set(
        meta_train["language"].astype(str)
        + "||"
        + meta_train["label_name"].astype(str)
        + "||"
        + meta_train["speaker_id"].astype(str)
    )

    # create unique test speaker keys
    test_speaker_keys = set(
        meta_test["language"].astype(str)
        + "||"
        + meta_test["label_name"].astype(str)
        + "||"
        + meta_test["speaker_id"].astype(str)
    )

    # find speakers present in both sets
    speaker_overlap = train_speaker_keys & test_speaker_keys

    # show speaker overlap count
    print("Speaker overlap:", len(speaker_overlap))

    # show examples when overlap exists
    if len(speaker_overlap) > 0:
        print("Example overlapping speaker keys:")
        print(list(speaker_overlap)[:10])

    # stop when speaker leakage is found
    assert len(speaker_overlap) == 0, "Speaker leakage detected."

In [ ]:
# run cross-lingual isolated-feature experiments

# store all cross-lingual results
crosslingual_results = []

# run each language setup
for setup in CROSS_LINGUAL_SETUPS:
    train_languages = setup["train_languages"]
    test_language = setup["test_language"]
    setup_name = setup["setup_name"]

    # evaluate each isolated feature
    for feature_name in FEATURE_NAMES:

        # build train and test data
        dataset_obj = build_crosslingual_train_test_isolated(
            mono_isolated=mono_isolated,
            feature_name=feature_name,
            train_languages=train_languages,
            test_language=test_language,
        )

        # check for data leakage
        check_crosslingual_no_leakage(
            dataset_obj=dataset_obj,
            train_languages=train_languages,
            test_language=test_language,
        )

        X_train = dataset_obj["X_train"]
        y_train = dataset_obj["y_train"]
        X_test = dataset_obj["X_test"]
        y_test = dataset_obj["y_test"]

        # evaluate with pca
        res_pca = evaluate_train_test(
            experiment_name=f"crosslingual_{setup_name}_{feature_name}_pca",
            feature_name=feature_name,
            language=test_language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=True,
            pca_components=0.95,
            random_state=42,
        )


        res_pca["training_setup"] = "crosslingual_train_two_languages"
        res_pca["train_languages"] = "+".join(train_languages)
        res_pca["test_language"] = test_language
        res_pca["setup_name"] = setup_name

        # save pca results
        crosslingual_results.append(res_pca)

        # evaluate without pca
        res_no_pca = evaluate_train_test(
            experiment_name=f"crosslingual_{setup_name}_{feature_name}_no_pca",
            feature_name=feature_name,
            language=test_language,
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            use_pca=False,
            pca_components=None,
            random_state=42,
        )

        res_no_pca["training_setup"] = "crosslingual_train_two_languages"
        res_no_pca["train_languages"] = "+".join(train_languages)
        res_no_pca["test_language"] = test_language
        res_no_pca["setup_name"] = setup_name

        # save no pca results
        crosslingual_results.append(res_no_pca)



In [ ]:
# run cross-lingual all-feature-fusion experiments

# run each cross-lingual setup
for setup in CROSS_LINGUAL_SETUPS:
    train_languages = setup["train_languages"]
    test_language = setup["test_language"]
    setup_name = setup["setup_name"]

    # build fused train and test data
    dataset_obj = build_crosslingual_train_test_fused(
        mono_fused=mono_fused,
        train_languages=train_languages,
        test_language=test_language,
    )

    # check for data leakage
    check_crosslingual_no_leakage(
        dataset_obj=dataset_obj,
        train_languages=train_languages,
        test_language=test_language,
    )


    X_train = dataset_obj["X_train"]
    y_train = dataset_obj["y_train"]
    X_test = dataset_obj["X_test"]
    y_test = dataset_obj["y_test"]

    # evaluate fused features with pca
    res_pca = evaluate_train_test(
        experiment_name=f"crosslingual_{setup_name}_all_features_fused_pca",
        feature_name="all_features_fused",
        language=test_language,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        use_pca=True,
        pca_components=0.95,
        random_state=42,
    )


    res_pca["training_setup"] = "crosslingual_train_two_languages"
    res_pca["train_languages"] = "+".join(train_languages)
    res_pca["test_language"] = test_language
    res_pca["setup_name"] = setup_name

    # save pca results
    crosslingual_results.append(res_pca)

    # evaluate fused features without pca
    res_no_pca = evaluate_train_test(
        experiment_name=f"crosslingual_{setup_name}_all_features_fused_no_pca",
        feature_name="all_features_fused",
        language=test_language,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        use_pca=False,
        pca_components=None,
        random_state=42,
    )


    res_no_pca["training_setup"] = "crosslingual_train_two_languages"
    res_no_pca["train_languages"] = "+".join(train_languages)
    res_no_pca["test_language"] = test_language
    res_no_pca["setup_name"] = setup_name

    # save no pca results
    crosslingual_results.append(res_no_pca)



In [ ]:
# combine and save cross-lingual results

# combine all cross-lingual result tables
crosslingual_results_df = pd.concat(
    crosslingual_results,
    ignore_index=True
)

# define the output csv path
CROSSLINGUAL_RESULTS_PATH = PROJECT / "none_interpretable_model_results_crosslingual_train_two_test_heldout_language.csv"

# save the combined results
crosslingual_results_df.to_csv(CROSSLINGUAL_RESULTS_PATH, index=False)


print("Saved cross-lingual results to:", CROSSLINGUAL_RESULTS_PATH)
print("Results shape:", crosslingual_results_df.shape)

print("\nColumns:")
print(crosslingual_results_df.columns.tolist())

display(
    crosslingual_results_df.sort_values(
        # rank by balanced accuracy first
        by=["balanced_accuracy", "f1", "roc_auc", "accuracy"],
        ascending=False
    ).head(50)
)
